# COMP 3610 Assignment 4: MLOps & Model Deployment

## Part 1: Experiment Tracking with MLFlow

In this section, I trained multiple machine learning models and track their performance using MLflow.
The goal is to compare models based on metrics such as MAE, RMSE, and R², and select the best-performing model.

### Task 1.1 : MLFlow Setup & Experiment Logging

In [24]:
# importing all necessary imports
import polars as pl
import pandas as pd
import numpy as np
import requests
import json
import os
from pathlib import Path

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
import matplotlib.pyplot as plt
from tabulate import tabulate


import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import joblib

print("All imports successful ")

All imports successful 


In [25]:
yellow_taxi_parquet_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet'
taxi_lookup_csv_url     = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

raw_data_dir = Path("data/raw")
raw_data_dir.mkdir(parents=True, exist_ok=True)

# download parquet
parquet_path = Path('data/raw/yellow_taxi.parquet')
if parquet_response := requests.get(yellow_taxi_parquet_url):
    if not parquet_path.exists():
        with open(parquet_path, 'wb') as f:
            f.write(parquet_response.content)
        print("Parquet file saved ")
    else:
        print("Parquet file already exists. Skipping...")

# download lookup CSV
csv_path = Path('data/raw/taxi_lookup.csv')
csv_response = requests.get(taxi_lookup_csv_url)
if csv_response.status_code == 200:
    if not csv_path.exists():
        with open(csv_path, 'wb') as f:
            f.write(csv_response.content)
        print("CSV file saved ")
    else:
        print("CSV file already exists. Skipping...")

Parquet file already exists. Skipping...
CSV file already exists. Skipping...


In [26]:
yellow_taxi_data = pl.read_parquet('data/raw/yellow_taxi.parquet')
taxi_zone_data   = pl.read_csv('data/raw/taxi_lookup.csv')

print(f"Loaded {len(yellow_taxi_data):,} rows")

expected_columns = {
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "PULocationID", "DOLocationID", "passenger_count",
    "trip_distance", "fare_amount", "total_amount", "payment_type"
}
for col in expected_columns:
    if col not in yellow_taxi_data.columns:
        raise ValueError(f"{col} not found!")
print("All required columns present ")

yellow_taxi_data.head()

Loaded 2,964,624 rows
All required columns present 


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


In [27]:
initial_row_count = len(yellow_taxi_data)
print(f"Initial rows: {initial_row_count:,}")

# drop nulls in critical columns
critical_columns = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "PULocationID", "DOLocationID", "fare_amount"
]
yellow_taxi_data = yellow_taxi_data.drop_nulls(critical_columns)

# demove bad fares, distances, and reversed times
yellow_taxi_data = yellow_taxi_data.filter(
    (pl.col("trip_distance") > 0) &
    (pl.col("fare_amount") > 0) &
    (pl.col("fare_amount") <= 500)
)
yellow_taxi_data = yellow_taxi_data.filter(
    pl.col("tpep_dropoff_datetime") > pl.col("tpep_pickup_datetime")
)

# dredit card trips only (tip_amount is only reliable for these)
yellow_taxi_data = yellow_taxi_data.filter(pl.col('payment_type') == 1)

final_row_count = len(yellow_taxi_data)
print(f"After cleaning: {final_row_count:,} rows")
print(f"Removed: {initial_row_count - final_row_count:,} rows")

Initial rows: 2,964,624
After cleaning: 2,298,347 rows
Removed: 666,277 rows


In [28]:

def weekend(num: int) -> bool:
    if num == 5 or num == 6:
        return True
    return False

def day(num: int) -> int:
    return num - 1

yellow_taxi_data = yellow_taxi_data.with_columns([
    pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour'),
    pl.col('tpep_pickup_datetime').dt.weekday()
      .map_elements(day, return_dtype=pl.Int32).alias('pickup_day_of_week'),
])

yellow_taxi_data = yellow_taxi_data.with_columns([
    pl.col('pickup_day_of_week')
      .map_elements(weekend, return_dtype=pl.Boolean).alias('is_weekend'),
])

# trip features
yellow_taxi_data = yellow_taxi_data.with_columns([
    ((pl.col('tpep_dropoff_datetime') - pl.col('tpep_pickup_datetime'))
     .dt.total_seconds() / 60).alias('trip_duration_minutes')
])

yellow_taxi_data = yellow_taxi_data.with_columns([
    pl.when(pl.col("trip_duration_minutes") <= 0)
      .then(0)
      .otherwise(pl.col("trip_distance") / pl.col("trip_duration_minutes"))
      .alias('trip_speed_mph'),
    pl.col('trip_distance').log1p().alias('log_trip_distance'),
])

#fare features
yellow_taxi_data = yellow_taxi_data.with_columns([
    pl.when(pl.col('trip_distance') > 0)
      .then(pl.col('fare_amount') / pl.col('trip_distance'))
      .otherwise(0).alias('fare_per_mile'),
    pl.when(pl.col('trip_duration_minutes') > 0)
      .then(pl.col('fare_amount') / pl.col('trip_duration_minutes'))
      .otherwise(0).alias('fare_per_minute'),
])

#zone features
yellow_taxi_data = yellow_taxi_data.join(
    taxi_zone_data.select(['LocationID', 'Borough']).rename(
        {'LocationID': 'PULocationID', 'Borough': 'pickup_borough'}),
    on='PULocationID', how='left'
)
yellow_taxi_data = yellow_taxi_data.join(
    taxi_zone_data.select(['LocationID', 'Borough']).rename(
        {'LocationID': 'DOLocationID', 'Borough': 'dropoff_borough'}),
    on='DOLocationID', how='left'
)

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# encoding pickup_borough
encoded = encoder.fit_transform(yellow_taxi_data.select('pickup_borough').to_numpy())
encoded_pickup = pl.DataFrame(
    encoded,
    schema=encoder.get_feature_names_out(['pickup_borough']).tolist()
)
yellow_taxi_data = yellow_taxi_data.hstack(encoded_pickup)

# encoding dropoff_borough
encoded = encoder.fit_transform(yellow_taxi_data.select('dropoff_borough').to_numpy())
encoded_dropoff = pl.DataFrame(
    encoded,
    schema=encoder.get_feature_names_out(['dropoff_borough']).tolist()
)
yellow_taxi_data = yellow_taxi_data.hstack(encoded_dropoff)


yellow_taxi_data = yellow_taxi_data.with_columns([
    pl.when((pl.col('fare_amount') * 0.20) > pl.col('tip_amount'))
      .then(0).otherwise(1).alias('high_tip')
])

print(f"Feature engineering complete.  Shape: {yellow_taxi_data.shape}")

Feature engineering complete.  Shape: (2298347, 46)


In [29]:
pd_data = yellow_taxi_data.to_pandas()

empty_cols    = pd_data.columns[pd_data.isna().all()].tolist()
datetime_cols = pd_data.select_dtypes(include='datetime').columns.tolist()
cols_to_drop  = empty_cols + datetime_cols

pd_data = pd_data.drop(columns=cols_to_drop)
pd_data = pd_data.drop(columns=['total_amount', 'pickup_borough', 'dropoff_borough'],
                        errors='ignore')

print(f"Remaining columns ({len(pd_data.columns)}): {pd_data.columns.tolist()}")
print(f"Shape: {pd_data.shape}")

# split
target_var = ['tip_amount']
X    = pd_data.drop(columns=target_var)
y = pd_data['tip_amount']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Remaining columns (41): ['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'congestion_surcharge', 'Airport_fee', 'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'trip_duration_minutes', 'trip_speed_mph', 'log_trip_distance', 'fare_per_mile', 'fare_per_minute', 'pickup_borough_Bronx', 'pickup_borough_Brooklyn', 'pickup_borough_EWR', 'pickup_borough_Manhattan', 'pickup_borough_N/A', 'pickup_borough_Queens', 'pickup_borough_Staten Island', 'pickup_borough_Unknown', 'dropoff_borough_Bronx', 'dropoff_borough_Brooklyn', 'dropoff_borough_EWR', 'dropoff_borough_Manhattan', 'dropoff_borough_N/A', 'dropoff_borough_Queens', 'dropoff_borough_Staten Island', 'dropoff_borough_Unknown', 'high_tip']
Shape: (2298347, 41)


In [30]:
numeric_features = [
    col for col in X.columns
    if X[col].dtype in ['int8', 'int32', 'int64', 'float64', 'bool']
]
categorical_features = [
    col for col in X.columns
    if X[col].dtype == 'object'
]

print(f"Numeric  ({len(numeric_features)}): {numeric_features}")
print(f"Categorical ({len(categorical_features)}): {categorical_features}")

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer,     numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

Numeric  (39): ['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tolls_amount', 'improvement_surcharge', 'congestion_surcharge', 'Airport_fee', 'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'trip_duration_minutes', 'trip_speed_mph', 'log_trip_distance', 'fare_per_mile', 'fare_per_minute', 'pickup_borough_Bronx', 'pickup_borough_Brooklyn', 'pickup_borough_EWR', 'pickup_borough_Manhattan', 'pickup_borough_N/A', 'pickup_borough_Queens', 'pickup_borough_Staten Island', 'pickup_borough_Unknown', 'dropoff_borough_Bronx', 'dropoff_borough_Brooklyn', 'dropoff_borough_EWR', 'dropoff_borough_Manhattan', 'dropoff_borough_N/A', 'dropoff_borough_Queens', 'dropoff_borough_Staten Island', 'dropoff_borough_Unknown', 'high_tip']
Categorical (1): ['store_and_fwd_flag']


### Training models

The following code trains different models and logs their performance metrics to MLflow for comparison.

In [31]:
# Point the notebook at your local MLflow server
mlflow.set_tracking_uri("http://localhost:5000")
print("MLflow tracking URI:", mlflow.get_tracking_uri())

# Create (or retrieve) the experiment required by the assignment
mlflow.set_experiment("taxi-tip-prediction")
print("Experiment 'taxi-tip-prediction' is ready ")

MLflow tracking URI: http://localhost:5000
Experiment 'taxi-tip-prediction' is ready 


In [32]:
def compute_metrics(model, X, y):
    y_pred = model.predict(X)
    mae = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)
    return mae, rmse, r2

In [34]:

with mlflow.start_run(run_name="linear_regression"):
    mlflow.set_tag("model_type", "LinearRegression")
    mlflow.set_tag("dataset_version", "yellow_taxi_2024_01")

    linear_regression_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ])

    linear_regression_pipeline.fit(X_train, y_train)
    mae, rmse, r2 = compute_metrics(
        linear_regression_pipeline,
        X_test,
        y_test
    )

    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(linear_regression_pipeline, "model")

C:\Users\maris.MARISHEL\Desktop\MLOps_and_Model_Deployment\.venv\Lib\site-packages\_distutils_hack\__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
C:\Users\maris.MARISHEL\Desktop\MLOps_and_Model_Deployment\.venv\Lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


![Linear Regression Image 1](screenshots/linear_reg_pic1.png)
![Linear Regression Image 2](screenshots/linear_reg_pic2.png)

In [35]:
with mlflow.start_run(run_name="random_forest_regression"):
    mlflow.set_tag("model_type",      "RandomForestRegressor")
    mlflow.set_tag("dataset_version", "yellow_taxi_2024_01")

    random_forest_regressor_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=50,
            random_state=42,
            n_jobs=-1,
            max_depth=10
        ))
    ])
    random_forest_regressor_pipeline.fit(X_train, y_train)
    mae, rmse, r2 = compute_metrics(random_forest_regressor_pipeline, X_test, y_test)


    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(random_forest_regressor_pipeline, "model")



C:\Users\maris.MARISHEL\Desktop\MLOps_and_Model_Deployment\.venv\Lib\site-packages\_distutils_hack\__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
C:\Users\maris.MARISHEL\Desktop\MLOps_and_Model_Deployment\.venv\Lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


![Random Forest Regression Image 1](screenshots/random_forest_reg_pic1.png)
![Random Forest Regression Image 2](screenshots/random_forest_reg_pic2.png)

### Task 1.2 : Model Comparison & Registry

Based on the MLflow results, the best model is selected using evaluation metrics.
This model is then saved along with metadata for use in the API.

![MAE](screenshots/mae.png)
![R2](screenshots/r2.png)
![RSME](screenshots/rsme.png)

In [36]:
#register best model
result = mlflow.register_model(
    model_uri="runs:/34067009f6f5494b98d736435c0eb695/model",
    name="best_model"
)

client = MlflowClient()

metrics = {
    "mae": 0.6469,
    "rmse": 1.6308,
    "r2": 0.8261
}

client.update_model_version(
    name="best_model",
    version= result.version,
    description=(
        f"Best Assignment 2 regression model for tip prediction. "
        f"MAE={metrics['mae']}, RMSE={metrics['rmse']}, R2={metrics['r2']}"
    )
)


os.makedirs("models", exist_ok=True)

# save metrics alone
with open("models/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

feature_names = X.columns.tolist()
model_metadata = {
    "model_name": "best_model",
    "model_version": str(result.version),
    "feature_names": feature_names,
    "training_metrics": metrics
}

with open("models/model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(model_metadata, f, indent=2)

print("Metrics saved to models/metrics.json")
print("Model metadata saved to models/model_metadata.json")


Registered model 'best_model' already exists. Creating a new version of this model...
2026/04/18 17:53:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: best_model, version 5


Metrics saved to models/metrics.json
Model metadata saved to models/model_metadata.json


Created version '5' of model 'best_model'.


In [37]:
loaded_model = mlflow.sklearn.load_model(f"models:/best_model/1")

sample_input = X_test.iloc[[0]]
sample_pred = loaded_model.predict(sample_input)[0]

print("Sample input:")
display(sample_input)

print("Predicted tip_amount:", round(float(sample_pred), 2))
print("Actual tip_amount:", round(float(y_test.iloc[0]), 2))

joblib.dump(loaded_model, "models/best_model.pkl")
print("Model saved to models/best_model.pkl ")

Sample input:


,VendorID,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,...,pickup_borough_Unknown,dropoff_borough_Bronx,dropoff_borough_Brooklyn,dropoff_borough_EWR,dropoff_borough_Manhattan,dropoff_borough_N/A,dropoff_borough_Queens,dropoff_borough_Staten Island,dropoff_borough_Unknown,high_tip
1826092,2,1,5.42,1,N,79,36,1,28.2,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1


Predicted tip_amount: 7.07
Actual tip_amount: 8.3
Model saved to models/best_model.pkl 


## Part 2: Model Serving with FastAPI

### Task 2.1: API Design & Implementation

FastAPI application was created to serve predictions from the trained model.
The API loads the trained model and exposes endpoints for prediction.

# Swagger UI tests

## / route

![route](screenshots/route-1.png)
![route2](screenshots/route-2.png)

## /predict route


![predict1](screenshots/predict1.png)
![predict2](screenshots/predict2.png)

## / predict / batch


![predict/batch](screenshots/predictbatch.png)
![predict/batch2](screenshots/predictbatch2.png)

## / health


![health](screenshots/health.png)
![health](screenshots/health2.png)

## /model / info


![model-info](screenshots/modelinfo.png)
![model-info](screenshots/modelinfo2.png)

### Task 2.3: API Testing

### PyTest Status

![tests](screenshots/tests.png)


# Part 3: Containerization with Docker

### Task 3.1: Dockerfile & Image Building

## Building Docker image

![docker-image](screenshots/docker-image.png)

## Running Docker Container

![docker-image-run](screenshots/docker-image-run.png)


## Verifying the API is accessible after running container


![docker-image-curl](screenshots/docker-image-curl.png)


# Docker Full WorkFlow

## Docker compose up


![docker-compose](screenshots/docker-compose-build.png)
![docker-compose2](screenshots/docker-compose-build2.png)

## 3 Predictions


![prediction1](screenshots/prediction1.png)
![prediction2](screenshots/prediction2.png)
![prediction3](screenshots/prediction3.png)


## Docker Compose down

![docker-compose-down](screenshots/docker-compose-down.png)

## Docker Container Size

![docker-container-size](screenshots/docker-container-size.png)

## Docker Images

![docker-images](screenshots/docker-images.png)